In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [6]:
import torch
import datasets
import transformers
import huggingface_hub

from datasets import load_dataset

print("PyTorch:", torch.__version__)
print("Datasets:", datasets.__version__)
print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Datasets: 4.0.0
Transformers: 5.13.1
Hugging Face Hub: 1.23.0
PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [3]:
classifier = pipeline("sentiment-analysis")

text = "I absolutely loved this movie."

result = classifier(text)

print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998776912689209}]


In [4]:
texts = [
    "I absolutely loved this movie.",
    "This was the worst movie I have ever seen.",
    "The movie was fantastic and the acting was brilliant.",
    "I hated every minute of this film."
]

results = classifier(texts)

for text, result in zip(texts, results):
    print("Text:", text)
    print("Prediction:", result["label"])
    print("Confidence:", round(result["score"], 4))
    print("-" * 50)

Text: I absolutely loved this movie.
Prediction: POSITIVE
Confidence: 0.9999
--------------------------------------------------
Text: This was the worst movie I have ever seen.
Prediction: NEGATIVE
Confidence: 0.9998
--------------------------------------------------
Text: The movie was fantastic and the acting was brilliant.
Prediction: POSITIVE
Confidence: 0.9999
--------------------------------------------------
Text: I hated every minute of this film.
Prediction: NEGATIVE
Confidence: 0.9995
--------------------------------------------------


In [9]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [10]:
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))

test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 2000
Testing samples: 500
